# GRPO-train Qwen2.5-1.5B on `exploitable_poker`

Runs end-to-end on a Colab A100/H100. Switch to a smaller config (drop `gpu_memory_utilization` to ~0.30) for T4.


## 1. Clone the glue repo and run setup

In [ ]:
!nvidia-smi || echo 'No GPU detected.'
%cd /content
![ -d clbench-verifiers ] || git clone https://github.com/sr-networks/clbench-verifiers.git
%cd clbench-verifiers
!bash scripts/colab_setup.sh

## 2. Quick env smoke test (no GPU)

Build the env, drive a few turns with random actions, and confirm rewards flow through.

In [ ]:
import asyncio
from clbench_verifiers.env import build_clbench_env

env = build_clbench_env(
    task_name='exploitable_poker',
    task_kwargs={'num_instances': 2, 'opponent_policy': 'calling_station', 'seed': 0},
    max_instances_per_rollout=1,
)

async def smoke():
    state = {'messages': []}
    await env.setup_state(state)
    print('System prompt:', state['messages'][0]['content'][:200], '...')
    print('First user prompt:', state['messages'][-1]['content'][:300], '...')
    # Pretend the model said something invalid → expect a re-prompt.
    state['messages'].append({'role': 'assistant', 'content': 'I will FOLD.'})
    next_msgs = await env.env_response(state['messages'], state)
    print('After invalid action:', next_msgs[0]['content'][:300])

await smoke()

## 3. Train

In [ ]:
!clbv-train --config configs/poker_qwen2_5_1_5b.toml

## 4. Evaluate the trained checkpoint via CLBench

In [ ]:
!clbv-eval --checkpoint ./checkpoints/poker_qwen2_5_1_5b/final \
    --task exploitable_poker --schedule quick_test